## **Prerequisites**

The code provided is intended to be run within the Jupyter Notebook platform. If you are using another IDE or platform for running the code, you may need to make minor adjustments.

Here, we would like to highlight some prerequisites that should be followed to ensure the code runs smoothly.

1- To run this notebook, you will need the following Python libraries installed: 

**{**pandas**, **matplotlib**, **seaborn**, **scikit-learn**, **ftfy**, **emoji**, **tensorflow**, **sentence-transformers**, **nltk**}**

For your convenience, we have included the `pip install` commands for these libraries, commented out in the cell below. 
Simply uncomment the relevant lines and run the cell to install them.

In [ ]:
!pip install pandas
!pip install matplotlib
!pip install scikit-learn
!pip install ftfy
!pip install emoji
!pip install tensorflow
!pip install nltk
!pip install sentencepiece

2- All necessary libraries for running the following cells are included in one cell below for import.

In [ ]:
# Import libraries

# --- Standard Library Imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import html
import nltk
import string
import ftfy
import emoji
import torch
from transformers import BertTokenizer, BertModel

# --- Scikit-learn Imports ---
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    f1_score,
    hamming_loss,
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve
)
from sklearn.pipeline import Pipeline

# --- NLTK Imports ---
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# --- PyTorch Imports ---
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertModel
from transformers import AlbertTokenizer, AlbertModel

3- In order to use the NLTK toolkit in this notebook, you need to download the following packages: 

**{**punkt**, **wordnet**, **stopwords**, **punkt_tab**}**

For your convenience, we have included the commands for downloading these packages, commented out in the cell below. Simply uncomment the relevant lines and run the cell to install them.

To check if the NLTK toolkit is installed, uncomment the last command in the cell below. The output should be as following:
['Hello', ',', 'I', "'m", 'fine', '.']

In [ ]:
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger')
print(word_tokenize("Hello, I'm fine."))

## Introduction:
**Overview:**
In this project, we focus on detecting emotions in text using multi-label classification. Each sentence in our **"Track-a"** dataset may express one or more emotions from a predefined set of **{joy, sadness, fear, anger, surprise}**. The goal is to train a model that can predict and classify these emotions in unseen sentences.

**Approaches:**

We will explore a few models in the upcoming sections, making small adjustments to the core code for each to observe their performance. The structure of our approach is as following:

1- Text Preprocessing, using a custom and NLTK method.

2- Defining several helpful functions that we are going to use in our model.

3- Machine Learning Model, utilizing Logistic Regresion classifier with TF-IDF (Term Frequency-Inverse Document Frequency) on both our custom and NLTK preprocessed text.

4- Deep Learning Model with BERT (Bidirectional Encoder Representations from Transformers)

**Evaluation:**

Each model's performance will be evaluated using micro-averaged-f1-score and Hamming Loss. Additionally, for a better visualization of the performances, we used "Receiver Operating Characteristic (ROC) curve".

## **References:**
In order to tackle a few challenges in our task, a number of resources was used:
1. [Datacamp's "Introduction to Natural Language Processing in Python"](https://app.datacamp.com/learn/skill-tracks/natural-language-processing-in-python)
2. [E. Mousavi's NLP Series, "Day 5: Handling Emojis"](https://medium.com/@ebimsv/nlp-series-day-5-handling-emojis-strategies-and-code-implementation-0f8e77e3a25c)
3. [Fixes text for you](https://ftfy.readthedocs.io/en/latest/)
4. [BERT-NLP-StackOverFlow-Prediction by Oren14](https://github.com/Oren14/BERT-NLP-StackOverflow-prediction/blob/main/predict-closed-questions-on-stackoverflow.ipynb)
5. [Datacamp's "Machine Learning Fundamentals with Python"](https://app.datacamp.com/learn/skill-tracks/machine-learning-fundamentals-with-python)

## **Acknowledgements:**
Grammarly was used for grammar and readability improvements of texts. ChatGPT was used for debugging, restructuring of some parts of codes, and most importantly for brainstorming. All AI-assisted parts are explicitly marked with comments.

In [ ]:
# --- If the CSV & Installation files are not in the default directory, change the folder using the command below: ---

# Load Dataset
file_path = "track-a.csv"
file = pd.read_csv(file_path)
df = pd.DataFrame(file)
df.info()
print("\n ------------------------------------------------------------- \n")
df.head()

## **Preprocessing Steps:**

To gain a better understanding, we looked at the first 200 rows, to see if there are anomalies, such as textual emojis (row #19), unnecessary use of special characters (row #35), tags (row #57), and HTML characters (row #185).

The following cell is the preprocessing section in order to have a clean and normal text for our model. We implemented two preprocessing approaches:
1. **Custom version**: Handles HTML, emojies, tags, and special characters. This part was just to research what libraries and techniques are mostly used. Then, we tried to somewhat make it useful for our own purposes.
2. **NLTK Preprocessing**: Uses Tokenization, lemmatization, and punctuation removal.

In [ ]:
# Preprocessing Steps without NLTK

# Fix Unicode and remove hidden characters
def fix_html(text):
    """ Fix HTML entities and Unicode issues"""
    
    text = ftfy.fix_text(text)
    text = html.unescape(text)
    return text

# -------------------------------------------------------------------

# Normalize text-reperesented emotions tokens
def normalize_emojies(text):
    """ Convert text emojis into text """
    
    text_emoji = {
        r';\s*\)': ' wink_face',
        r':\s*\)': ' smile_face',
        r':\s*\(': ' sad_face',
        r':\s*d': ' laugh_face',
        r':\s*p': ' playful_face',
        r':\s*/': ' skeptical_face',
        r':\s*o': ' surprised_face'
    }
    
    for pattern, replacement in text_emoji.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
        
    text = emoji.demojize(text, delimiters=(" ", " "))
    return text
    
# -------------------------------------------------------------------

# Additional Clean ups
def cleanup(text):
    """ Cleaning up special characters and formatting """
    
    # turning slash tags such as /chills into textual tag
    text = re.sub(r'(?<!\w)/([a-z]+)', r'\1_tag', text)
    # (AI Improvement idea) Turn Word/Word into word or word in order to not be mistaken as a tag
    text = re.sub(r'(\b\w+)\s*/\s*(\w+\b)', r'\1 or \2', text)

    # Remove non-necessary special characters in text
    text = re.sub(r"[`$]", " ", text)
    text = re.sub(r"''+", " ", text)
    text = re.sub(r'“|”|"', '', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# -------------------------------------------------------------------

# All-in one preprocessing function without NLTK
def preprocess_text_custom(text):
    text = text.lower()
    text = fix_html(text)
    text = normalize_emojies(text)
    text = cleanup(text)

    return text

# -------------------------------------------------------------------

# Applying preprocessing on the dataset
df['custom_text'] = df['text'].apply(preprocess_text_custom)

# A few examples of how texts look like with our custom preprocessing
check_1 = df['custom_text'][0:10]
print(f"Custom Preprocessing output: \n {check_1}")

In [ ]:
# Preprocessing using NLTK

lemmatizer = WordNetLemmatizer()

def nltk_preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)

    tokens = [lemmatizer.lemmatize(t) for  t in tokens if t not in string.punctuation]

    return ' '.join(tokens)

# Applying preprocessing on the dataset with NLTK
df['NLTK_text'] = df['text'].apply(nltk_preprocess)

# A few examples of how texts look like with our preprocessing
check_2 = df['NLTK_text'][0:10]
print(f"NLTK Preprocess Output: \n {check_2}")

### **Helpful Functions**

We initially implemented all our models, calling these functions multiple times. Ultimately, we restructured everything into one cell for easier and faster processing.

In [ ]:
# The debugging and restructuring of this cell were done using ChatGPT
def optimal_thresholds(y_true, y_pred_prob, data_format='dataframe'):
    """ Find Optimal Thresholds for each emotion class 
    Args:
    y_true: True labels (DataFrame for TF-IDF, list for BERT)
    y_pred_prob: Predicted Probabilities
    data_format: 'dataframe' for TF-IDF models, 'list' for BERT models
    """
    optimal_thresholds_list = []
    for i, emotion in enumerate(emotion_labels):
        opt_thresh = 0.5
        opt_f1 = 0
        
        # For TF-IDF Model
        if data_format == 'dataframe':
            true_labels = y_true.iloc[:,i]
        # For Bert Model
        else:
            true_labels = [label[i] for label in y_true]
            
        for thr in np.arange(0.05, 0.9, 0.05):
            predicts = (y_pred_prob[:,i] >= thr).astype(int)
            f1 = f1_score(true_labels, predicts, zero_division = 0)
            
            if f1 > opt_f1:
                opt_f1 = f1
                opt_thresh = thr

        optimal_thresholds_list.append(opt_thresh)
        print(f"{emotion_labels[i]}: {opt_thresh:.2f}(F1:{opt_f1:.3f})")
        
    return optimal_thresholds_list

def plot_roc_curves(y_true, y_scores, title="ROC Curves", data_format='dataframe'):
    """
    Plot ROC curves for all emotion classes
    
    Args:
        y_true: True labels 
        y_scores: Predicted probabilities
        title: Plot title
        data_format: 'dataframe' for TF-IDF, 'list' for BERT
    """
    plt.figure(figsize=(6, 10))
    
    for i, emotion in enumerate(emotion_labels):
        if data_format == 'dataframe':
            true_labels = y_true.iloc[:, i]
        else:
            true_labels_array = np.array(y_true)
            true_labels = true_labels_array[:, i]
            
        fpr, tpr, _ = roc_curve(true_labels, y_scores[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f'{emotion} (AUC = {roc_auc:.2f})')
    
    plt.plot([0, 1], [0, 1], '--', color='gray')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.0])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()

def evaluate_my_model(y_true, y_pred_prob, model_name, data_format='dataframe'):
    """
    Evaluation function for all models

    Args: 
        y_true: True Labels
        y_pred_prob: Predicted Probabilities
        model_name: Name of the model
        data_format: 'dataframe' for TF-IDF, 'list' for BERT
    """
    print(f"\n{'='*50}")
    print(f"EVALUATING {model_name.upper()}")
    print(f"{'='*50}")

    # Finding & Applying optimal thresholds
    thresholds = optimal_thresholds(y_true, y_pred_prob, data_format)
    y_pred = np.zeros_like(y_pred_prob, dtype=int)
    for i, threshold in enumerate(thresholds):
        y_pred[:, i] = (y_pred_prob[:, i] >= threshold).astype(int)

    if data_format == 'list':
        y_true_array = np.array(y_true)
    else:
        y_true_array = y_true.values

    # Micro-Average-F1-Score and Hamming Loss evaluation techniques
    micro_f1 = f1_score(y_true_array, y_pred, average='micro')
    hamming = hamming_loss(y_true_array, y_pred)

    print(f"\nResults:")
    print(f"Micro F1-score: {micro_f1:.4f}")
    print(f"Hamming Loss: {hamming:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_true_array, y_pred, target_names=emotion_labels, zero_division=0))

    # Plot ROC curves
    plot_roc_curves(y_true, y_pred_prob, f"{model_name} ROC Curves", data_format)

    return thresholds

## **First Model: Logistic Regression with TF-IDF Tokenization**

In this section, we use **TF-IDF vectorization** to convert our preprocessed text into features, and then train a multi-label **"Logistic Regression"** classifier using a One-vs-Rest Strategy. Logistic Regression is chosen due to its simplicity, speed, and reliable performance on smaller datasets and in text classification tasks. Additionally, the GridSearch is applied for hyperparameter tuning.

In the subsequent cells, we evaluate the model's performance using the micro-averaged-f1-score and Hamming Loss, providing a good understanding of our model's effectiveness across all emotions. Additionally, to gain a visualized evaluation, we use ROC Curves for each label.

In [ ]:
# Define Data and labels for the model
emotion_labels = ['anger', 'fear', 'joy', 'sadness', 'surprise']
X = df['custom_text']
y = df[emotion_labels]

# Split into Train / Validation / Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=52)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=52)

print(f"Train: {len(X_train)} | Validation: {len(X_val)} | Test: {len(X_test)}")

# Model Pipeline using Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=12000, ngram_range=(1, 2), analyzer='word', stop_words='english', lowercase=False)),
    ('clf', OneVsRestClassifier(LogisticRegression(solver='liblinear', class_weight='balanced')))
])


# Parameter Tuning with Grid Search 
param_grid = {
    'tfidf__max_features': [8000, 12000],
    'tfidf__ngram_range': [(1, 2), (1, 3)],
    'clf__estimator__C': [0.5, 1.0, 2.0]
}
grid = GridSearchCV(pipeline, param_grid=param_grid, scoring='f1_micro', cv=3, verbose=1)
grid.fit(X_train, y_train)

print("Best hyperparameters:", grid.best_params_)

# Prediction Step
y_pred_prob = grid.predict_proba(X_test)

optimal_threshold = evaluate_my_model(y_test, y_pred_prob, "TF-IDF Custom Preprocessing")

# Sentence Emotion Prediction Command using Logistic Regression + TF-IDF
def predict_via_Logist_tfidf(text, model=grid.best_estimator_, thresholds=optimal_threshold):
    vectorized = model.named_steps['tfidf'].transform([text])
    probs = model.named_steps['clf'].predict_proba(vectorized)[0]
    preds = []
    for i, thr in enumerate(thresholds):
        preds.append(1 if probs[i] >= thr else 0)
        
    return dict(zip(emotion_labels, preds))

In [ ]:
# Logistic Regression TF-IDF Example Test

test_sentences = [
    "I am so excited about this new opportunity!",
    "I'm really scared about the upcoming exam.",
    "This makes me so angry and frustrated!",
    "I feel so sad and lonely today.",
    "Wow, I never expected this to happen!",
    "I wish there were no wars in the world...",
    "I wish ice creams were as tasty as people say."
]

for sentence in test_sentences:
    result = predict_via_Logist_tfidf(sentence, grid.best_estimator_, optimal_threshold)
    print(f"Text: '{sentence}'")
    print(f"Predicted emotions: {result}\n")

In [ ]:
# Now the same code, just using the NLTK preprocessing

# Data and labels for the model
emotion_labels = ['anger', 'fear', 'joy', 'sadness', 'surprise']
X = df['NLTK_text'] # Only the input here is changed <--
y = df[emotion_labels]

# Split into Train / Validation / Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=52)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=52)

print(f"Train: {len(X_train)} | Validation: {len(X_val)} | Test: {len(X_test)}")

# Model Pipeline using Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=12000, ngram_range=(1, 2), analyzer='word', stop_words=None, lowercase=False)),
    ('clf', OneVsRestClassifier(LogisticRegression(solver='liblinear', class_weight='balanced')))
])


# Parameter Tuning with Grid Search 
param_grid = {
    'tfidf__max_features': [8000, 12000],
    'tfidf__ngram_range': [(1, 2), (1, 3)],
    'clf__estimator__C': [0.5, 1.0, 2.0]
}
grid_nltk = GridSearchCV(pipeline, param_grid=param_grid, scoring='f1_micro', cv=3, verbose=1)
grid_nltk.fit(X_train, y_train)

print("Best hyperparameters:", grid_nltk.best_params_)

# Prediction Step
y_pred_prob = grid_nltk.predict_proba(X_test)

optimal_threshold_nltk = evaluate_my_model(y_test, y_pred_prob, "TF-IDF NLTK Preprocessing")


# New Sentence Prediction Commands
def predict_via_Log_NLTK_tfidf(text, model=grid.best_estimator_, thresholds=optimal_threshold_nltk):
    vectorized = model.named_steps['tfidf'].transform([text])
    probs = model.named_steps['clf'].predict_proba(vectorized)[0]
    preds = []
    for i, thr in enumerate(thresholds):
        preds.append(1 if probs[i] >= thr else 0)
        
    return dict(zip(emotion_labels, preds))

In [ ]:
# Logistic Regression NLTK + TF-IDF Example Test
test_sentences = [
    "I am so excited about this new opportunity!",
    "I'm really scared about the upcoming exam.",
    "This makes me so angry and frustrated!",
    "I feel so sad and lonely today.",
    "Wow, I never expected this to happen!",
    "I wish there were no wars in the world...",
    "I wish ice creams were as tasty as people say.",
    "I just received some incredibly disappointing news, and I'm struggling to keep it together."
]

for sentence in test_sentences:
    result = predict_via_Log_NLTK_tfidf(sentence, grid.best_estimator_, optimal_threshold)
    print(f"Text: '{sentence}'")
    print(f"Predicted emotions: {result}\n")

## **Second Model: BERT**

For our second model, we implement **BERT (Bidirectional Encoder Representation from Transformers)** for the emotion classification task. While our previous TF-IDF model served as a good baseline, we wanted to explore the implementation and performance of a more advanced approach like BERT.

This model serves primarily as an exploration into new techniques, with its development leveraging resources from YouTube, Stack Overflow, Github repositories, and AI tools for brainstorming, coding, debugging and enhancing the structure of our implementation.

In [ ]:
# Data and labels for the model
emotion_labels = ['anger', 'fear', 'joy', 'sadness', 'surprise']
X = df['custom_text']
y = df[emotion_labels]

df['labels']= y.values.tolist()

X_train, X_temp, y_train, y_temp = train_test_split(X, df['labels'], test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Bert Tokenizer
tokenizer_custom = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Tokenizer Function
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.float)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }


train_dataset = EmotionDataset(X_train.tolist(), y_train.tolist(), tokenizer_custom)
val_dataset = EmotionDataset(X_val.tolist(), y_val.tolist(), tokenizer_custom)
test_dataset = EmotionDataset(X_test.tolist(), y_test.tolist(), tokenizer_custom)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

# Bert Model Function
def create_Dbert_custom_model(num_labels=5):
    class DistilBERT(nn.Module):
        def __init__(self):
            super().__init__()
            self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
            self.dropout = nn.Dropout(0.3)
            self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        
        def forward(self, input_ids, attention_mask):
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            pooled_output = outputs.last_hidden_state[:, 0]
            pooled_output = self.dropout(pooled_output)
            return self.classifier(pooled_output)
    
    return DistilBERT()

# Training Function
# The debugging and restructuring of this function were done using ChatGPT
def train_distilbert(model, train_loader, val_loader=None, epochs=3, lr=2e-5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")
    
    return model


# Prediction Function
def predict_bert(model, dataloader, batch_size=8):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_predictions.append(probs)

    return np.vstack(all_predictions)

In [ ]:
# Prediction Function
def predict_emotion_via_BERT(text, model, tokenizer, thresholds, max_len=128):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    # Tokenize input text
    encoded = tokenizer(text, truncation=True, padding='max_length', 
                       max_length=max_len, return_tensors='pt')
    
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)
    
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()[0]
        
        preds = []
        for i, threshold in enumerate(thresholds):
            preds.append(1 if probs[i] >= threshold else 0)
    
    return dict(zip(emotion_labels, preds))

In [ ]:
# Create and train model
model_custom = create_Dbert_custom_model()
trained_model_custom = train_distilbert(model_custom, train_loader, val_loader, epochs=3, lr=2e-5)

# Get predictions
y_pred_prob = predict_bert(trained_model_custom, test_loader)

# Evaluate
optimal_threshold_Bert_custom = evaluate_my_model(y_test.tolist(), y_pred_prob, "BERT Custom Preprocessing", 'list')

In [ ]:
test_sentences = [
    "I am so excited about this new opportunity!",
    "I'm really scared about the upcoming exam.",
    "This makes me so angry and frustrated!",
    "I feel so sad and lonely today.",
    "Wow, I never expected this to happen!",
    "I wish there were no wars in the world...",
    "I am not doing well."
]

for sentence in test_sentences:
    result = predict_emotion_via_BERT(sentence, trained_model_custom, tokenizer_custom, optimal_threshold_Bert_custom)
    print(f"Text: '{sentence}'")
    print(f"Predicted emotions: {result}\n")

In [ ]:
# Again, the exact same code as before, just using the NLTK preprocessing steps we did before.

# Data and labels for the model
emotion_labels = ['anger', 'fear', 'joy', 'sadness', 'surprise']
X = df['NLTK_text']
y = df[emotion_labels]

df['labels']= y.values.tolist()

X_train, X_temp, y_train, y_temp = train_test_split(X, df['labels'], test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Bert Tokenizer
tokenizer_nltk = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Tokenizer Function
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.float)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }


train_dataset = EmotionDataset(X_train.tolist(), y_train.tolist(), tokenizer_nltk)
val_dataset = EmotionDataset(X_val.tolist(), y_val.tolist(), tokenizer_nltk)
test_dataset = EmotionDataset(X_test.tolist(), y_test.tolist(), tokenizer_nltk)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

# Bert Model Function
def create_Dbert_nltk_model(num_labels=5):
    class DistilBERT(nn.Module):
        def __init__(self):
            super().__init__()
            self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
            self.dropout = nn.Dropout(0.3)
            self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        
        def forward(self, input_ids, attention_mask):
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            pooled_output = outputs.last_hidden_state[:, 0]  # CLS token
            pooled_output = self.dropout(pooled_output)
            return self.classifier(pooled_output)
    
    return DistilBERT()

# Training Function
# The debugging and restructuring of this function were done using ChatGPT
def train_distilbert(model, train_loader, val_loader=None, epochs=3, lr=2e-5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")
    
    return model

# Prediction Function
def predict_bert_NLTK(model, dataloader, batch_size=8):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_predictions.append(probs)

    return np.vstack(all_predictions)

In [ ]:
# Prediction Function
def predict_emotion_via_NLTK_BERT(text, model, tokenizer, thresholds, max_len=128):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    # Tokenize input text
    encoded = tokenizer(text, truncation=True, padding='max_length', 
                       max_length=max_len, return_tensors='pt')
    
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)
    
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()[0]
        
        preds = []
        for i, threshold in enumerate(thresholds):
            preds.append(1 if probs[i] >= threshold else 0)
    
    return dict(zip(emotion_labels, preds))

In [ ]:
# Create and train model
model_nltk = create_Dbert_nltk_model()
trained_model_nltk = train_distilbert(model_nltk, train_loader, val_loader, epochs=3, lr=2e-5)

# Get predictions
y_pred_prob = predict_bert_NLTK(trained_model_nltk, test_loader)

# Evaluate
optimal_threshold_bert_nltk = evaluate_my_model(y_test.tolist(), y_pred_prob, "BERT NLTK Preprocessing", 'list')

In [ ]:
test_sentences = [
    "I am so excited about this new opportunity!",
    "I'm really scared about the upcoming exam.",
    "This makes me so angry and frustrated!",
    "I feel so sad and lonely today.",
    "Wow, I never expected this to happen!",
    "I wish there were no wars in the world...",
    "I am not doing well."
]

for sentence in test_sentences:
    result = predict_emotion_via_NLTK_BERT(sentence, trained_model_nltk, tokenizer_nltk, optimal_threshold_bert_nltk)
    print(f"Text: '{sentence}'")
    print(f"Predicted emotions: {result}\n")

### **Simpler BERT Model**

Since we needed to create a checkpoint and present the trained model, we discovered that the pickle file we intended to showcase would be approximately 250MB. To address this, we simplified our model to reduce its size, but this resulted in a reduction in its accuracy.

In the cell below, we have the same code as the previous BERT model; however, we are now using a lighter version called ALBERT.

In [ ]:
# Data and labels for the model
emotion_labels = ['anger', 'fear', 'joy', 'sadness', 'surprise']
X = df['custom_text']
y = df[emotion_labels]

df['labels']= y.values.tolist()

X_train, X_temp, y_train, y_temp = train_test_split(X, df['labels'], test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Bert Tokenizer
albert_tokenizer_custom = AlbertTokenizer.from_pretrained('albert-base-v2')

# Tokenizer Function
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.float)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }


train_dataset = EmotionDataset(X_train.tolist(), y_train.tolist(), albert_tokenizer_custom)
val_dataset = EmotionDataset(X_val.tolist(), y_val.tolist(), albert_tokenizer_custom)
test_dataset = EmotionDataset(X_test.tolist(), y_test.tolist(), albert_tokenizer_custom)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

# Albert Model Function
def create_albert_custom_model(num_labels=5):
    class ALBERTClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.bert = AlbertModel.from_pretrained('albert-base-v2')
            self.dropout = nn.Dropout(0.3)
            self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        
        def forward(self, input_ids, attention_mask):
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            pooled_output = outputs.last_hidden_state[:, 0]
            pooled_output = self.dropout(pooled_output)
            return self.classifier(pooled_output)
    
    return ALBERTClassifier()

# Training Function
# The debugging and restructuring of this function were done using ChatGPT
def train_albert(model, train_loader, val_loader=None, epochs=3, lr=2e-5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")
    
    return model


# Prediction Function
def predict_albert(model, dataloader, batch_size=8):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_predictions.append(probs)

    return np.vstack(all_predictions)

# Prediction Function
def predict_emotion_via_albert(text, model, tokenizer, thresholds, max_len=128):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    # Tokenize input text
    encoded = tokenizer(text, truncation=True, padding='max_length', 
                       max_length=max_len, return_tensors='pt')
    
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)
    
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()[0]
        
        preds = []
        for i, threshold in enumerate(thresholds):
            preds.append(1 if probs[i] >= threshold else 0)
    
    return dict(zip(emotion_labels, preds))

In [ ]:
# Create and train model
model_albert = create_albert_custom_model()
albert_trained_model = train_albert(model_albert, train_loader, val_loader, epochs=3, lr=2e-5)

# Get predictions
y_pred_prob = predict_albert(albert_trained_model, test_loader)

# Evaluate
optimal_threshold_albert_custom = evaluate_my_model(y_test.tolist(), y_pred_prob, "ALBERT Custom Preprocessing", 'list')

### **Predict my CSV**

This section allows you to upload a test CSV file, execute the selected model, and generate an output CSV file based on the chosen model.

In the final cell, please specify the first input as the path of the CSV file, the column name containing the sentences, and the desired output file name.

As this was a last-minute edit, the template for the functions below was AI-generated but edited to match our models.


### **References**
[Python - How to Save and Load ML Models](https://www.kaggle.com/code/prmohanty/python-how-to-save-and-load-ml-models)

In [ ]:
def process_csv_emotions(csv_file_path, text_column_name, output_csv_path=None, model=grid.best_estimator_, thresholds=optimal_threshold):
    """
    Process a CSV file and add emotion prediction columns
    
    Args:
        csv_file_path: Path to input CSV file
        text_column_name: Name of the column containing text to analyze
        output_csv_path: Path to save output CSV (if None, returns DataFrame)
        model: Trained model to use for predictions
        thresholds: Optimal thresholds for predictions
    
    Returns:
        DataFrame with original data + emotion prediction columns
    """
    # Read the CSV file
    df_new = pd.read_csv(csv_file_path)
    
    print(f"Processing {len(df_new)} rows from {csv_file_path}")
    print(f"Analyzing text from column: '{text_column_name}'")
    
    # Check if the text column exists
    if text_column_name not in df_new.columns:
        raise ValueError(f"Column '{text_column_name}' not found. Available columns: {list(df_new.columns)}")
    
    # Initialize emotion columns
    for emotion in emotion_labels:
        df_new[f'predicted_{emotion}'] = 0
    
    # Process each row
    for idx, row in df_new.iterrows():
        text = str(row[text_column_name])  # Convert to string in case of NaN
        
        # Skip empty or NaN texts
        if pd.isna(text) or text.strip() == '' or text.lower() == 'nan':
            continue
            
        # Get emotion predictions
        emotions = predict_via_Logist_tfidf(text, model, thresholds)
        
        # Fill in the emotion columns
        for emotion, prediction in emotions.items():
            df_new.at[idx, f'predicted_{emotion}'] = prediction

    
    # Save to file if output path provided
    if output_csv_path:
        df_new.to_csv(output_csv_path, index=False)
        print(f"\n Results saved to: {output_csv_path}")
    
    print("Processing complete!\n")
    return df_new


# --------------------------------------------------------------------------------
def process_csv_emotions_NLTK(csv_file_path, text_column_name, output_csv_path=None, model=grid_nltk.best_estimator_, thresholds=optimal_threshold_nltk):
    """
    Process a CSV file using TF-IDF + NLTK model
    """
    # Read the CSV file
    df_new = pd.read_csv(csv_file_path)
    
    print(f"Processing {len(df_new)} rows from {csv_file_path}")
    print(f"Analyzing text from column: '{text_column_name}' (TF-IDF NLTK Model)")
    
    # Check if the text column exists
    if text_column_name not in df_new.columns:
        raise ValueError(f"Column '{text_column_name}' not found. Available columns: {list(df_new.columns)}")
    
    # Initialize emotion columns
    for emotion in emotion_labels:
        df_new[f'predicted_{emotion}_nltk'] = 0
    
    for idx, row in df_new.iterrows():
        text = str(row[text_column_name])
        
        # Skip empty or NaN texts
        if pd.isna(text) or text.strip() == '' or text.lower() == 'nan':
            continue
            
        # Get emotion predictions
        emotions = predict_via_Log_NLTK_tfidf(text, model, thresholds)
        
        # Fill in the emotion columns
        for emotion, prediction in emotions.items():
            df_new.at[idx, f'predicted_{emotion}_nltk'] = prediction
    
    # Save to file if output path provided
    if output_csv_path:
        df_new.to_csv(output_csv_path, index=False)
        print(f"\n Results saved to: {output_csv_path}")
    
    print("NLTK processing complete!\n")
    return df_new

# --------------------------------------------------------------------------------
def process_csv_emotions_BERT_custom(csv_file_path, text_column_name, output_csv_path=None, 
                                   model=trained_model_custom, tokenizer=tokenizer_custom, 
                                   thresholds=optimal_threshold_Bert_custom):
    """
    Process a CSV file using BERT Custom model
    """
    # Read the CSV file
    df_new = pd.read_csv(csv_file_path)
    
    print(f"Processing {len(df_new)} rows from {csv_file_path}")
    print(f"Analyzing text from column: '{text_column_name}' (BERT Custom Model)")
    
    # Check if the text column exists
    if text_column_name not in df_new.columns:
        raise ValueError(f"Column '{text_column_name}' not found. Available columns: {list(df_new.columns)}")
    
    # Initialize emotion columns
    for emotion in emotion_labels:
        df_new[f'predicted_{emotion}_bert_custom'] = 0
    
    # Process each row
    print("Generating BERT-based predictions...")
    
    for idx, row in df_new.iterrows():
        text = str(row[text_column_name])
        
        # Skip empty or NaN texts
        if pd.isna(text) or text.strip() == '' or text.lower() == 'nan':
            continue
            
        # Get emotion predictions
        emotions = predict_emotion_via_BERT(text, model, tokenizer, thresholds)
        
        # Fill in the emotion columns
        for emotion, prediction in emotions.items():
            df_new.at[idx, f'predicted_{emotion}_bert_custom'] = prediction
    
    
    # Save to file if output path provided
    if output_csv_path:
        df_new.to_csv(output_csv_path, index=False)
        print(f"\nResults saved to: {output_csv_path}")
    
    print("BERT Custom processing complete!\n")
    return df_new

# --------------------------------------------------------------------------------
def process_csv_emotions_BERT_NLTK(csv_file_path, text_column_name, output_csv_path=None, 
                                 model=trained_model_nltk, tokenizer=tokenizer_nltk, 
                                 thresholds=optimal_threshold_bert_nltk):
    """
    Process a CSV file using BERT NLTK model
    """
    # Read the CSV file
    df_new = pd.read_csv(csv_file_path)
    
    print(f"Processing {len(df_new)} rows from {csv_file_path}")
    print(f"Analyzing text from column: '{text_column_name}' (BERT NLTK Model)")
    
    # Check if the text column exists
    if text_column_name not in df_new.columns:
        raise ValueError(f"Column '{text_column_name}' not found. Available columns: {list(df_new.columns)}")
    
    # Initialize emotion columns
    for emotion in emotion_labels:
        df_new[f'predicted_{emotion}_bert_nltk'] = 0
    
    # Process each row
    print("Generating BERT NLTK predictions...")
    
    for idx, row in df_new.iterrows():
        text = str(row[text_column_name])
        
        # Skip empty or NaN texts
        if pd.isna(text) or text.strip() == '' or text.lower() == 'nan':
            continue
            
        # Get emotion predictions
        emotions = predict_emotion_via_NLTK_BERT(text, model, tokenizer, thresholds)
        
        # Fill in the emotion columns
        for emotion, prediction in emotions.items():
            df_new.at[idx, f'predicted_{emotion}_bert_nltk'] = prediction
    
    # Save to file if output path provided
    if output_csv_path:
        df_new.to_csv(output_csv_path, index=False)
        print(f"\nResults saved to: {output_csv_path}")
    
    print("BERT NLTK processing complete!\n")
    return df_new

# ---------------------------------------------------------------------------------------------
def process_csv_emotions_albert(csv_file_path, text_column_name, output_csv_path=None, 
                                 model=albert_trained_model, tokenizer=albert_tokenizer_custom, 
                                 thresholds=optimal_threshold_albert_custom):
    """
    Process a CSV file using Albert and custom preprocessing
    """
    # Read the CSV file
    df_new = pd.read_csv(csv_file_path)
    
    print(f"Processing {len(df_new)} rows from {csv_file_path}")
    print(f"Analyzing text from column: '{text_column_name}' (Albert Model)")
    
    # Check if the text column exists
    if text_column_name not in df_new.columns:
        raise ValueError(f"Column '{text_column_name}' not found. Available columns: {list(df_new.columns)}")
    
    # Initialize emotion columns
    for emotion in emotion_labels:
        df_new[f'predicted_{emotion}_bert_nltk'] = 0
    
    # Process each row
    print("Generating Albert + custom predictions...")
    
    for idx, row in df_new.iterrows():
        text = str(row[text_column_name])
        
        # Skip empty or NaN texts
        if pd.isna(text) or text.strip() == '' or text.lower() == 'nan':
            continue
            
        # Get emotion predictions
        emotions = predict_emotion_via_albert(text, model, tokenizer, thresholds)
        
        # Fill in the emotion columns
        for emotion, prediction in emotions.items():
            df_new.at[idx, f'predicted_{emotion}_albert_custom'] = prediction
    
    # Save to file if output path provided
    if output_csv_path:
        df_new.to_csv(output_csv_path, index=False)
        print(f"\nResults saved to: {output_csv_path}")
    
    print("Albert + Custom processing complete!\n")
    return df_new

In [ ]:
result0 = process_csv_emotions("my_data.csv", "text", "output_custom.csv")
result1 = process_csv_emotions_NLTK("my_data.csv", "text", "output_nltk.csv")
result2 = process_csv_emotions_BERT_custom("my_data.csv", "text", "output_bert_custom.csv")
result3 = process_csv_emotions_BERT_NLTK("my_data.csv", "text", "output_bert_nltk.csv")
result4 = process_csv_emotions_albert("my_data.csv", "text", "output_albert_custom.csv")